In [1]:
!pip install -q sentence-transformers transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.7 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from pypdf import PdfReader
import faiss
import numpy as np

In [4]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.5 MB/s eta 0:00:00


In [5]:
!pip install -q reportlab
from reportlab.pdfgen import canvas

# Create a sample PDF file
pdf_filename = "sample_document.pdf"
c = canvas.Canvas(pdf_filename)
c.drawString(100, 750, "Hello, this is a test PDF.")
c.drawString(100, 730, "You can extract this text using pypdf.")
c.save()


In [6]:
reader = PdfReader("sample_document.pdf")

text = ""

for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        text += extracted + "\n"

In [7]:
chunk_size = 500

chunks = [
    text[i:i+chunk_size]
    for i in range(0, len(text), chunk_size)
]

print("Chunks:", len(chunks))

Chunks: 1


In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

In [10]:
question = "What is Artificial Intelligence?"

query_embedding = model.encode([question])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=3
)

In [12]:
context = "\n".join(
    chunks[i]
    for i in indices[0]
)

print(context)

Hello, this is a test PDF.
You can extract this text using pypdf.


Hello, this is a test PDF.
You can extract this text using pypdf.


Hello, this is a test PDF.
You can extract this text using pypdf.




In [13]:
generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

response = generator(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

print(response[0]["generated_text"])

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_th


Context:
Hello, this is a test PDF.
You can extract this text using pypdf.


Hello, this is a test PDF.
You can extract this text using pypdf.


Hello, this is a test PDF.
You can extract this text using pypdf.



Question:
What is Artificial Intelligence?

Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What is Artificial Intelligence?
Answer:
What


In [ ]:
while True:

    question = input("Ask a question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    query_embedding = model.encode([question])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k=3
    )

    context = "\n".join(
        chunks[i]
        for i in indices[0]
    )

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=100
    )

    print("\nAnswer:\n")
    print(response[0]["generated_text"])
    print("-" * 50)

Ask a question (type 'exit' to quit): what is genai


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:


Context:
Hello, this is a test PDF.
You can extract this text using pypdf.


Hello, this is a test PDF.
You can extract this text using pypdf.


Hello, this is a test PDF.
You can extract this text using pypdf.



Question:
what is genai

Answer:
Genai is a non-native language in the Japanese language. The Japanese language is a modern language that is used in many languages in the United States.
I am not saying that it is easy to understand. I have written a lot of books about genai, and I am not saying that it is easy to understand. I have written a lot of books about genai, and I am not saying that it is easy to understand. I have written a lot of books about genai,
--------------------------------------------------
Ask a question (type 'exit' to quit): What is Machine Learning


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:


Context:
Hello, this is a test PDF.
You can extract this text using pypdf.


Hello, this is a test PDF.
You can extract this text using pypdf.


Hello, this is a test PDF.
You can extract this text using pypdf.



Question:
What is Machine Learning

Answer:
This is a test PDF.
What are Machine Learning?
Answer:
This is a test PDF.
What are Machine Learning?
Answer:
This is a test PDF.
What are Machine Learning?
Answer:
This is a test PDF.
What are Machine Learning?
Answer:
This is a test PDF.
What are Machine Learning?
Answer:
This is a test PDF.
What are Machine Learning?
Answer:
This is a test
--------------------------------------------------
